In [1]:
!pip install shap transformers torch -q

In [2]:
import shap
import torch
import numpy as np
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

In [3]:
import os
os.makedirs('./saved_model', exist_ok=True)

for filename in ['model.safetensors', 'config.json',
                 'tokenizer.json', 'tokenizer_config.json']:
    if os.path.exists(f'/content/{filename}'):
        import shutil
        shutil.move(f'/content/{filename}', f'./saved_model/{filename}')
        print(f"✅ Moved {filename}")

print("\nContents:", os.listdir('./saved_model'))

✅ Moved model.safetensors
✅ Moved config.json
✅ Moved tokenizer.json
✅ Moved tokenizer_config.json

Contents: ['model.safetensors', 'tokenizer.json', 'config.json', 'tokenizer_config.json']


In [4]:
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL_PATH = './saved_model'

tokenizer  = AutoTokenizer.from_pretrained(MODEL_PATH)
model      = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()

classifier = pipeline(
    'text-classification',
    model=model,
    tokenizer=tokenizer,
    return_all_scores=True,
    device=0 if torch.cuda.is_available() else -1
)

# Quick test
test = classifier("Scientists confirm miracle cure hidden by Big Pharma!")
print("✅ Model loaded!")
print("Test prediction:", test)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

✅ Model loaded!
Test prediction: [{'label': 'FAKE', 'score': 0.9992102384567261}]


In [5]:
explainer = shap.Explainer(classifier)
print("✅ SHAP explainer ready!")

✅ SHAP explainer ready!


In [6]:
fake_text = ["BREAKING: Government secretly poisoning water supply! "\
             "Scientists CONFIRM what they've been hiding for decades. "\
             "Share before this gets deleted!!!"]

shap_values = explainer(fake_text)
shap.plots.text(shap_values[0, :, 0])

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [00:39, 39.39s/it]               


In [7]:
real_text = ["According to a WHO report published on Monday, global "\
             "vaccination rates have increased by 12 percent in developing "\
             "nations over the past year, driven by international funding."]

shap_values_real = explainer(real_text)
shap.plots.text(shap_values_real[0, :, 1])

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [00:37, 37.66s/it]               


In [11]:
my_text = ["Indian government announces new AI policy to boost technology sector, with funding of 10000 crore rupees allocated for research and development over next 5 years."]

shap_values_mine = explainer(my_text)
shap.plots.text(shap_values_mine[0, :, 1])

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 2it [00:35, 35.64s/it]               
